# Roboflow Tree Detection Test

This notebook runs a Roboflow object detection model on the saved City of Vienna Orthofoto / ViennaGIS image.

It prints the predictions, saves them to `outputs/roboflow_predictions.json`, draws bounding boxes, and saves the result to `outputs/tree_detection_result.png`.

The predictions JSON file is used by notebook 03 to create tree GeoJSON.

The input image is an aerial orthophoto, not normal satellite imagery. The notebook reads the default Vienna coordinates and bbox from `data/vienna_orthofoto_test_metadata.json`.

In [ ]:
# Import the libraries we need.
# os lets us read environment variables like ROBOFLOW_API_KEY.
# sys lets us add the project folder to Python's import path.
# Path helps us build file paths that work on different operating systems.
# json lets us save predictions to a file for the next notebook.
# pprint prints dictionaries in a more readable way.
import json
import os
import sys
from pathlib import Path
from pprint import pprint

from dotenv import load_dotenv

In [ ]:
# Find the project root folder.
# If this notebook is opened from the notebooks/ folder, the project root is one level up.
# If it is opened from the project root, the current folder is already the project root.
current_dir = Path.cwd()

if (current_dir / ".env").exists():
    project_root = current_dir
else:
    project_root = current_dir.parent

# Add the project root to Python's import path.
# This lets the notebook import code from backend/app/detection.
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

# Load variables from the .env file into Python's environment.
# override=True matters in Jupyter: an old kernel can keep the previous model ID in memory.
load_dotenv(project_root / ".env", override=True)

In [ ]:
# Read the Roboflow API key and model ID.
# We do not print the API key, because API keys should stay private.
# ROBOFLOW_MODEL_ID should look like: project-name/version-number
roboflow_api_key = os.getenv("ROBOFLOW_API_KEY")
roboflow_model_id = os.getenv("ROBOFLOW_MODEL_ID")

# Stop early with a clear message if either value is missing.
if not roboflow_api_key:
    raise ValueError("ROBOFLOW_API_KEY is missing. Add it to your .env file first.")

if not roboflow_model_id:
    raise ValueError("ROBOFLOW_MODEL_ID is missing. Add it to your .env file first.")

print(f"Using Roboflow model: {roboflow_model_id}")

In [ ]:
# Import the reusable helper functions from the backend.
# run_tree_detection sends the image to Roboflow.
# draw_bounding_boxes creates a copy of the image with boxes drawn on it.
from backend.app.detection.roboflow_detector import draw_bounding_boxes, run_tree_detection
from backend.app.imagery.vienna_orthofoto import image_pixel_to_lonlat

In [ ]:
# Set the input and output paths.
# The input image should already exist from notebooks/01b_vienna_orthofoto_test.ipynb.
image_path = project_root / "data" / "vienna_orthofoto_test.png"
metadata_path = project_root / "data" / "vienna_orthofoto_test_metadata.json"
output_path = project_root / "outputs" / "tree_detection_result.png"
predictions_path = project_root / "outputs" / "roboflow_predictions.json"

# Make sure the outputs folder exists before saving files.
output_path.parent.mkdir(parents=True, exist_ok=True)

# Stop early if the Vienna orthophoto image has not been downloaded yet.
if not image_path.exists():
    raise FileNotFoundError(
        f"Missing input image: {image_path}. Run notebooks/01b_vienna_orthofoto_test.ipynb first."
    )

if not metadata_path.exists():
    raise FileNotFoundError(
        f"Missing georeferencing metadata: {metadata_path}. Run notebooks/01b_vienna_orthofoto_test.ipynb first."
    )


# Load the default Vienna center, bbox, CRS, and image size created by notebook 01b.
# This metadata lets us convert Roboflow pixel detections into real lon/lat coordinates.
metadata = json.loads(metadata_path.read_text(encoding="utf-8"))
bbox = metadata["bbox"]
image_width = metadata["image_width"]
image_height = metadata["image_height"]
default_lat = metadata["center"]["lat"]
default_lon = metadata["center"]["lon"]

print(f"Default Vienna center: lat={default_lat}, lon={default_lon}")
print(f"Image CRS: {metadata['crs']}")
print(f"Image size: {image_width} x {image_height}")

In [ ]:
# Run inference on the saved Vienna orthophoto image.
# confidence=40 means Roboflow will only return predictions with at least 40% confidence.
# overlap=30 controls how overlapping boxes are filtered.
result = run_tree_detection(
    image_path=str(image_path),
    api_key=roboflow_api_key,
    model_id=roboflow_model_id,
    confidence=40,
    overlap=30,
)

# Roboflow returns predictions in a list under the "predictions" key.
predictions = result.get("predictions", [])

# Add lon/lat for each detection center using the Vienna orthophoto metadata.
# Roboflow x/y coordinates are image pixels; the metadata maps those pixels back to Vienna.
for prediction in predictions:
    lon, lat = image_pixel_to_lonlat(
        px=float(prediction["x"]),
        py=float(prediction["y"]),
        bbox=(bbox["min_x"], bbox["min_y"], bbox["max_x"], bbox["max_y"]),
        image_width=image_width,
        image_height=image_height,
    )
    prediction["lon"] = lon
    prediction["lat"] = lat

print(f"Number of predictions: {len(predictions)}")
pprint(predictions)

# Save predictions plus input metadata so notebook 03 can detect stale outputs after coordinate changes.
prediction_output = {
    "model_id": roboflow_model_id,
    "input_image": str(image_path),
    "input_metadata": metadata,
    "predictions": predictions,
}
predictions_path.write_text(json.dumps(prediction_output, indent=2), encoding="utf-8")
print(f"Saved predictions to: {predictions_path}")

In [ ]:
# Draw bounding boxes on the original image and save the result.
# Each prediction should contain x, y, width, and height values.
saved_output_path = draw_bounding_boxes(
    image_path=str(image_path),
    predictions=predictions,
    output_path=str(output_path),
)

print(f"Saved detection result to: {saved_output_path}")

In [ ]:
# Display the image with bounding boxes in the notebook.
# This makes it easy to visually inspect whether the tree detections look reasonable.
import matplotlib.pyplot as plt
from PIL import Image

result_image = Image.open(saved_output_path)

plt.figure(figsize=(8, 8))
plt.imshow(result_image)
plt.axis("off")
plt.title("Roboflow Tree Detection - Vienna Orthofoto")
plt.show()